In [ ]:
## imports
# import plotnine
# from plotnine import *
import random

import numpy as np
import pandas as pd

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

from datetime import datetime, timedelta

## Load data

In [ ]:
## load data on 2020 crimes in DC
df = dc_crim_2020 = pd.read_csv(
    "https://opendata.arcgis.com/datasets/f516e0dd7b614b088ad781b0c4002331_2.csv"
)

## create report_dt column
df["report_dt"] = pd.to_datetime(df.REPORT_DAT)

## Warm-up Demo

In [ ]:
%%time
for i in range(df.shape[0]):
    r = df.iloc[i]
    r.X + r.Y

CPU times: user 936 ms, sys: 4.1 ms, total: 940 ms
Wall time: 944 ms


In [ ]:
%%time
for i,r in df.iterrows():
    r.X + r.Y

CPU times: user 369 ms, sys: 5.28 ms, total: 374 ms
Wall time: 379 ms


In [ ]:
%%time
df.apply(lambda r: r.X + r.Y, axis = 1)

CPU times: user 154 ms, sys: 2.85 ms, total: 157 ms
Wall time: 158 ms


0        534560.0
1        535553.0
2        534219.0
3        534863.0
4        533968.0
           ...   
27927    543971.0
27928    538173.0
27929    539531.0
27930    538757.0
27931    532367.0
Length: 27932, dtype: float64

In [ ]:
%%time
## Super fast, but only works with built-in numpy functions.
df.X + df.Y

CPU times: user 227 μs, sys: 42 μs, total: 269 μs
Wall time: 260 μs


0        534560.0
1        535553.0
2        534219.0
3        534863.0
4        533968.0
           ...   
27927    543971.0
27928    538173.0
27929    539531.0
27930    538757.0
27931    532367.0
Length: 27932, dtype: float64

# Practice

In [ ]:
## define crimes to look for and crimes to look within
## CCN is Central Complaint Number: https://go.mpdconline.com/GO/GO_401_01.pdf
CCN_examples = ["20165648", "20123250"]
C_Tar = C_Target = crimes_lookfor = df[df.CCN.astype(str).isin(CCN_examples)][
    ["CCN", "WARD", "OFFENSE", "report_dt"]
]
C_Oth = C_Other = other_crimes = df[~df.CCN.astype(str).isin(CCN_examples)]

## print crimes_lookfor
C_Tar.head()
# other_crimes.head()

,CCN,WARD,OFFENSE,report_dt
7198,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00
12139,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00


**Task**: we have two crimes we want to look for. We want to look in the remaining crime reports for crime reports that are:

- Located in the same ward as the two focal crimes
- Reported at the same time as the focal crime or up to 1000 minutes later (changed from slides which stated 20 mins since crime ids changed since last time so this long bandwidth helps us find matches!)

Solutions compare two ways to solve:

- Using a for loop
- Using a function

## 1. Loop approach

In [ ]:
## create empty container to store results
store_matches = {}

## loop through two example crimes
for i in range(C_Tar.shape[0]):  # same as shape

    ## extract row
    r = one_row = C_Tar.iloc[i]

    ## first, subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]

    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt + timedelta(minutes=1200)

    ### substep: use that to subset
    same_wards_sametime = same_wards[
        (same_wards.report_dt >= r.report_dt) & (same_wards.report_dt <= CUTOFF)
    ].copy()

    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime

## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

X         Y       CCN              REPORT_DAT  \
20165648 7131   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
         7132   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
         8089   398651.0  136899.0  20166039  2020/11/20 22:07:10+00   
         13609  400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
         13611  399489.0  137478.0  20165986  2020/11/20 22:17:27+00   

                            START_DATE                END_DATE  \
20165648 7131   2020/11/19 23:43:15+00                     NaN   
         7132   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
         8089   2020/11/20 17:30:16+00  2020/11/20 22:08:28+00   
         13609  2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
         13611  2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   

                                                     BLOCK  \
20165648 7131    600 - 669 BLOCK OF PENNSYLVANIA AVENUE SE   
         7132          600 - 699 BLOCK OF ORLEANS PLACE NE   
         8089   300 - 363 BLOCK OF MASSACHUSETTS AVENUE NW   
         13609              800 - 899 BLOCK OF H STREET NE   
         13611          1151 - 1199 BLOCK OF 1ST STREET NE   

                            OFFENSE  METHOD    SHIFT  ...  CENSUS_TRACT  \
20165648 7131           THEFT/OTHER  OTHERS      DAY  ...        6500.0   
         7132          THEFT F/AUTO  OTHERS      DAY  ...       10602.0   
         8089           THEFT/OTHER  OTHERS  EVENING  ...        5900.0   
         13609          THEFT/OTHER  OTHERS      DAY  ...        8402.0   
         13611  MOTOR VEHICLE THEFT  OTHERS  EVENING  ...       10603.0   

               VOTING_PRECINCT           BID    XBLOCK    YBLOCK   LATITUDE  \
20165648 7131      Precinct 89  CAPITOL HILL  400232.0  135255.0  38.885133   
         7132      Precinct 83           NaN  400233.0  137456.0  38.904961   
         8089     Precinct 143      DOWNTOWN  398651.0  136899.0  38.899942   
         13609     Precinct 82           NaN  400489.0  136927.0  38.900195   
         13611    Precinct 144          NOMA  399489.0  137478.0  38.905159   

                LONGITUDE   OBJECTID OCTO_RECORD_ID                 report_dt  
20165648 7131  -76.997326  981157643            NaN 2020-11-20 12:46:32+00:00  
         7132  -76.997314  981157644            NaN 2020-11-20 14:45:06+00:00  
         8089  -77.015552  981164991            NaN 2020-11-20 22:07:10+00:00  
         13609 -76.994363  981209755            NaN 2020-11-20 15:37:59+00:00  
         13611 -77.005891  981209757            NaN 2020-11-20 22:17:27+00:00  

[5 rows x 26 columns]

# 1.5 Iterrow Approach

In [ ]:
## create empty container to store results
store_matches = {}

## loop through two example crimes
for i, r in C_Tar.iterrows():  # same as

    ## subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]

    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt + timedelta(minutes=1200)

    ### substep: use that to subset
    same_wards_sametime = same_wards[
        (same_wards.report_dt >= r.report_dt) & (same_wards.report_dt <= CUTOFF)
    ].copy()

    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime

## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

## 2. Function approach

Practice rewriting the above loop as a function

In [ ]:
other_crimes

,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,...,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,OCTO_RECORD_ID,report_dt
0,402374.0,132186.0,10251445,2020/09/11 04:00:00+00,2010/09/02 04:00:00+00,2010/09/02 04:00:00+00,2300 - 2399 BLOCK OF AINGER PLACE SE,HOMICIDE,GUN,MIDNIGHT,...,7502.0,Precinct 134,NaN,402374.0,132186.0,38.857483,-76.972648,980886212,NaN,2020-09-11 04:00:00+00:00
1,397330.0,138223.0,16641,2020/01/28 05:14:39+00,2020/01/28 05:14:58+00,2020/01/28 05:15:02+00,1300 - 1399 BLOCK OF CORCORAN STREET NW,THEFT F/AUTO,OTHERS,MIDNIGHT,...,5001.0,Precinct 16,NaN,397330.0,138223.0,38.911866,-77.030785,980886214,NaN,2020-01-28 05:14:39+00:00
2,402411.0,131808.0,10147537,2020/12/21 05:00:00+00,2010/10/10 02:00:00+00,NaN,2300 - 2499 BLOCK OF HARTFORD STREET SE,HOMICIDE,OTHERS,MIDNIGHT,...,7408.0,Precinct 115,NaN,402411.0,131808.0,38.854078,-76.972223,980887143,NaN,2020-12-21 05:00:00+00:00
3,397920.0,136943.0,5370,2020/01/17 02:39:05+00,2020/01/11 02:48:51+00,2020/01/17 02:48:54+00,800 - 899 BLOCK OF 9TH STREET NW,THEFT F/AUTO,OTHERS,EVENING,...,5802.0,Precinct 129,DOWNTOWN,397920.0,136943.0,38.900337,-77.023979,980887228,NaN,2020-01-17 02:39:05+00:00
4,393681.0,140287.0,20158444,2020/11/05 22:10:59+00,2020/10/23 19:00:15+00,2020/10/30 21:00:38+00,3000 - 3199 BLOCK OF WISCONSIN AVENUE NW,THEFT/OTHER,OTHERS,EVENING,...,704.0,Precinct 28,NaN,393681.0,140287.0,38.930441,-77.072878,980907455,NaN,2020-11-05 22:10:59+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27927,398758.0,145213.0,20014291,2020/01/24 16:16:36+00,2020/01/24 15:20:16+00,2020/01/24 15:30:51+00,100 - 199 BLOCK OF CARROLL STREET NW,THEFT F/AUTO,OTHERS,DAY,...,1702.0,Precinct 63,NaN,398758.0,145213.0,38.974837,-77.014333,981478862,NaN,2020-01-24 16:16:36+00:00
27928,397323.0,140850.0,20015815,2020/01/27 00:15:35+00,2020/01/26 19:30:13+00,2020/01/26 19:31:48+00,3530 - 3545 BLOCK OF HOLMEAD PLACE NW,ASSAULT W/DANGEROUS WEAPON,GUN,EVENING,...,2900.0,Precinct 42,NaN,397323.0,140850.0,38.935531,-77.030876,981478863,NaN,2020-01-27 00:15:35+00:00
27929,397624.0,141907.0,20064390,2020/04/28 03:20:48+00,2020/04/26 16:00:38+00,2020/04/28 01:30:53+00,1100 - 1199 BLOCK OF ALLISON STREET NW,THEFT/OTHER,OTHERS,MIDNIGHT,...,2501.0,Precinct 48,NaN,397624.0,141907.0,38.945054,-77.027408,981478864,NaN,2020-04-28 03:20:48+00:00
27930,401191.0,137566.0,20064709,2020/04/28 21:24:04+00,2020/04/25 20:00:17+00,2020/04/28 13:30:26+00,1200 - 1299 BLOCK OF PENN STREET NE,THEFT F/AUTO,OTHERS,EVENING,...,8802.0,Precinct 77,NaN,401191.0,137566.0,38.905951,-76.986269,981478865,NaN,2020-04-28 21:24:04+00:00


In [ ]:
C_Tar

,CCN,WARD,OFFENSE,report_dt
7198,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00
12139,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00


### 2.1 define the function

In [41]:
cols = other_crimes.columns
store_matches_2 = {}


def find_related_crimes(r):
    filt = other_crimes.loc[other_crimes["WARD"] == r["WARD"]]
    filt = filt.loc[
        (filt["report_dt"] >= r["report_dt"])
        & (filt["report_dt"] <= r["report_dt"] + pd.Timedelta(minutes=1000))
    ]

    return filt

### 2.2 apply it to one of the focal crimes

In [42]:
%%time

C_Tar["matches"] = C_Tar.apply(find_related_crimes, axis=1)

CPU times: user 5.25 ms, sys: 4.04 ms, total: 9.29 ms
Wall time: 9 ms


### 2.3 Use apply to cover all the other focal crimes

In [43]:
C_Tar

,CCN,WARD,OFFENSE,report_dt,matches
7198,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00,X Y CCN ...
12139,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00,X Y CCN ...


In [46]:
%%time

store_matches_2 = {}

def find_related_crimes(r):
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    CUTOFF = r.report_dt + timedelta(minutes=1200)

    same_wards_sametime = same_wards[
        (same_wards.report_dt >= r.report_dt) &
        (same_wards.report_dt <= CUTOFF)
    ].copy()

    return same_wards_sametime

CPU times: user 3 μs, sys: 8 μs, total: 11 μs
Wall time: 14.1 μs


In [49]:
list(C_Tar["matches"])[1]

,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,...,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,OCTO_RECORD_ID,report_dt
870,395618.0,138388.0,20123422,2020/08/29 16:45:57+00,2020/08/26 22:00:29+00,2020/08/27 12:00:51+00,2200 - 2399 BLOCK OF DECATUR PLACE NW,THEFT F/AUTO,OTHERS,DAY,...,4100.0,Precinct 13,NaN,395618.0,138388.0,38.913346,-77.050526,980911088,NaN,2020-08-29 16:45:57+00:00
17651,396662.0,138429.0,20401318,2020/08/29 14:29:59+00,2020/08/28 20:55:00+00,2020/08/28 21:05:00+00,1724 - 1799 BLOCK OF 17TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,5302.0,Precinct 15,NaN,396662.0,138429.0,38.913720,-77.038489,981388845,NaN,2020-08-29 14:29:59+00:00
23220,396523.0,137976.0,20123389,2020/08/29 16:05:18+00,2020/08/28 22:00:23+00,2020/08/29 08:00:27+00,1700 - 1799 BLOCK OF P STREET NW,THEFT F/AUTO,OTHERS,DAY,...,5303.0,Precinct 15,NaN,396523.0,137976.0,38.909638,-77.040089,981436904,NaN,2020-08-29 16:05:18+00:00
23796,398098.0,136808.0,20123419,2020/08/29 17:15:19+00,2020/08/29 16:05:40+00,2020/08/29 16:08:33+00,700 - 799 BLOCK OF 7TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,5801.0,Precinct 129,DOWNTOWN,398098.0,136808.0,38.899121,-77.021926,981442283,NaN,2020-08-29 17:15:19+00:00
